In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 12.0 MB/s eta 0:00:00


In [5]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.1 MB/s eta 0:00:00


In [6]:
from PIL import Image
import imagehash
import pandas as pd
from pathlib import Path
import numpy as np
from tqdm import tqdm
import faiss

In [10]:
EMB_DIR = "/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/embeddings"

embeddings = np.load(f"{EMB_DIR}/clip_embeddings.npy").astype("float32")
meta = pd.read_csv(f"{EMB_DIR}/clip_embeddings_meta.csv")

index = faiss.read_index("/content/drive/MyDrive/copydays-ndid/Nagarjuna/ndid_faiss.index")

In [11]:
K = 10
CLIP_THRESHOLD = 0.90
PHASH_THRESHOLD = 5

In [13]:
def classify_duplicate_with_phash(query_idx):
    q = embeddings[query_idx].reshape(1, -1)
    sims, idxs = index.search(q, K)

    sims = sims[0][1:]     # remove self
    idxs = idxs[0][1:]

    best_clip_sim = sims[0]

    query_path = meta.iloc[query_idx]["image_path"]
    query_img = Image.open(query_path)
    query_hash = imagehash.phash(query_img)

    best_phash_dist = 64  # max possible

    for j in idxs:
        cand_path = meta.iloc[j]["image_path"]
        cand_img = Image.open(cand_path)
        cand_hash = imagehash.phash(cand_img)

        dist = query_hash - cand_hash
        best_phash_dist = min(best_phash_dist, dist)

        if dist <= PHASH_THRESHOLD:
            return "DUPLICATE (pHash)", best_clip_sim, dist, idxs.tolist()

    if best_clip_sim >= CLIP_THRESHOLD:
        return "DUPLICATE (CLIP)", best_clip_sim, best_phash_dist, idxs.tolist()

    return "NOT DUPLICATE", best_clip_sim, best_phash_dist, idxs.tolist()

In [14]:
query_idx = 123

decision, clip_sim, phash_dist, neighbors = classify_duplicate_with_phash(query_idx)

print("Decision:", decision)
print("Best CLIP similarity:", clip_sim)
print("Best pHash distance:", phash_dist)

Decision: DUPLICATE (pHash)
Best CLIP similarity: 0.9722934
Best pHash distance: 0
